In [1]:
import tensorflow as tf
import pathlib
import json

def train_transfer_learning_model_local():
    dataset_path = r"D:\CV_Project\flowers"

    data_dir = pathlib.Path(dataset_path)

    # Quick check to make sure the path exists
    if not data_dir.exists():
        print(f"Error: Could not find the folder at {data_dir}")
        return

    batch_size = 32
    img_height = 180
    img_width = 180

    print("Loading datasets...")
    # This automatically reads the subfolders as your class names
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="training", seed=123,
        image_size=(img_height, img_width), batch_size=batch_size)

    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir, validation_split=0.2, subset="validation", seed=123,
        image_size=(img_height, img_width), batch_size=batch_size)

    class_names = train_ds.class_names
    print(f"Found classes: {class_names}")

    # Save classes for the Streamlit app
    with open('class_names.json', 'w') as f:
        json.dump(class_names, f)

    num_classes = len(class_names)

    print("Loading MobileNetV2 Base Model...")
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(img_height, img_width, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    print("Building new Model Architecture...")
    model = tf.keras.Sequential([
        tf.keras.layers.Rescaling(1./127.5, offset=-1, input_shape=(img_height, img_width, 3)),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes)
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])

    print("Training Model (Transfer Learning)...")
    model.fit(train_ds, validation_data=val_ds, epochs=5)

    print("Saving model to 'flower_model.keras'...")
    model.save('flower_model.keras')
    print("Training complete!")

if __name__ == "__main__":
    train_transfer_learning_model_local()

Loading datasets...
Found 4317 files belonging to 5 classes.
Using 3454 files for training.
Found 4317 files belonging to 5 classes.
Using 863 files for validation.
Found classes: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
Loading MobileNetV2 Base Model...


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_41076\1955970108.py:39: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Building new Model Architecture...
Training Model (Transfer Learning)...
Epoch 1/5


C:\Users\KIIT0001\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


108/108 ━━━━━━━━━━━━━━━━━━━━ 207s 2s/step - accuracy: 0.3153 - loss: 1.7224 - val_accuracy: 0.4809 - val_loss: 1.3001
Epoch 2/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.5405 - loss: 1.1669 - val_accuracy: 0.6477 - val_loss: 0.9733
Epoch 3/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 346s 3s/step - accuracy: 0.6520 - loss: 0.9172 - val_accuracy: 0.7161 - val_loss: 0.8095
Epoch 4/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.7212 - loss: 0.7677 - val_accuracy: 0.7601 - val_loss: 0.7118
Epoch 5/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 99s 919ms/step - accuracy: 0.7522 - loss: 0.6800 - val_accuracy: 0.7879 - val_loss: 0.6476
Saving model to 'flower_model.keras'...
Training complete!


In [1]:
%%writefile app.py

import streamlit as st
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import json
from sklearn.cluster import KMeans
from rembg import remove

# 1. Load Model and Classes
@st.cache_resource
def load_model_and_classes():
    model = tf.keras.models.load_model('flower_model.keras')
    with open('class_names.json', 'r') as f:
        class_names = json.load(f)
    return model, class_names

# 2. Helper to find closest color name
COLORS = {
    "Red": (255, 0, 0), "Green": (0, 255, 0), "Blue": (0, 0, 255),
    "Yellow": (220, 148, 11), "Cyan": (0, 255, 255), "Magenta": (255, 0, 255),
    "White": (255, 255, 255), "Black": (0, 0, 0), "Orange": (255, 165, 0),
    "Purple": (128, 0, 128), "Pink": (255, 192, 203)
}

def get_color_name(rgb_tuple):
    min_dist = float('inf')
    closest_name = "Unknown"
    for name, color in COLORS.items():
        # Calculate Euclidean distance between the detected color and our predefined colors
        dist = sum((a - b) ** 2 for a, b in zip(rgb_tuple, color))
        if dist < min_dist:
            min_dist = dist
            closest_name = name
    return closest_name

# 3. Main Streamlit App
st.title("🌸 Flower Classification & Analysis App")
st.write("Upload an image of a flower to classify it, detect its dominant color, and view its color histogram.")

try:
    model, class_names = load_model_and_classes()
except Exception as e:
    st.error("Model not found. Please run `train_model.py` first to generate the model.")
    st.stop()

uploaded_file = st.file_uploader("Choose a flower image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Read the image
    image = Image.open(uploaded_file).convert('RGB')
    img_array = np.array(image)

    st.image(image, caption='Uploaded Image', use_column_width=True)

    st.markdown("---")
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("1. Classification")
        # Preprocess for CNN
        resized_img = tf.image.resize(img_array, [180, 180])
        input_arr = tf.expand_dims(resized_img, 0) # Create a batch

        # Predict
        predictions = model.predict(input_arr)
        score = tf.nn.softmax(predictions[0])
        predicted_class = class_names[np.argmax(score)]
        confidence = 100 * np.max(score)

        st.success(f"**Prediction:** {predicted_class}")
        st.info(f"**Confidence:** {confidence:.2f}%")

        st.subheader("2. Color Detection")

        with st.spinner("Removing background to isolate flower..."):
            # 1. Remove the background (this makes the background transparent)
            img_no_bg = remove(image)

            # 2. Convert to numpy array (it is now RGBA: Red, Green, Blue, Alpha/Transparency)
            img_no_bg_array = np.array(img_no_bg)

            # 3. Create a mask to ONLY select pixels that are NOT transparent
            # The 4th channel (index 3) is the Alpha/Transparency channel
            mask = img_no_bg_array[:, :, 3] > 0

            # 4. Grab only the RGB values of the actual flower, ignoring the background entirely
            flower_pixels = img_no_bg_array[mask][:, :3]

        # 5. Use KMeans on JUST the flower pixels
        if len(flower_pixels) > 0:
            kmeans = KMeans(n_clusters=1, n_init=10)
            kmeans.fit(flower_pixels)
            dominant_color = kmeans.cluster_centers_[0].astype(int)
            color_name = get_color_name(dominant_color)

            st.write(f"**Dominant Color (RGB):** {dominant_color}")
            st.write(f"**Closest Match:** {color_name}")
            # Display a color block
            st.markdown(f'<div style="width:50px;height:50px;background-color:rgb({dominant_color[0]},{dominant_color[1]},{dominant_color[2]}); border: 1px solid black;"></div>', unsafe_allow_html=True)

            # Optional: Show the background-free image to the user
            st.image(img_no_bg, caption="Isolated Flower", use_column_width=True)
        else:
            st.warning("Could not isolate the flower from the background.")

    with col2:
        st.subheader("3. Color Histogram")
        # Calculate and plot Histogram using OpenCV and Matplotlib
        fig, ax = plt.subplots()
        colors = ('r', 'g', 'b')
        for i, color in enumerate(colors):
            hist = cv2.calcHist([img_array], [i], None, [256], [0, 256])
            ax.plot(hist, color=color)
            ax.set_xlim([0, 256])
        ax.set_title("RGB Color Histogram")
        ax.set_xlabel("Pixel Intensity")
        ax.set_ylabel("Frequency")
        st.pyplot(fig)

Overwriting app.py
